# CS383: Data Science and Machine Learning
## Lecture 6 — Feature Engineering

---

## Part 1 — Why Feature Engineering

Every model we build starting next week expects its input as **numbers, on comparable scales**. Real
data almost never arrives that way — it arrives as category strings, wildly different numeric ranges, and
columns that were never designed with a model in mind. Feature engineering is the work of turning what you
collected into what a model can actually use, without quietly breaking anything along the way.

### Setup — NYC restaurant inspections

Same dataset and live-pull-with-fallback pattern from Lectures 4 and 5.

In [ ]:
import os
import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

SOCRATA_URL = "https://data.cityofnewyork.us/resource/43nn-pn8j.json"

try:
    raw_path = os.path.expanduser("~/shared-readwrite/restaurant_inspections_snapshot.csv")
    raw = pd.read_csv(raw_path).head(15000)
    raw["score"] = pd.to_numeric(raw["score"], errors="coerce")

    inspections_df = (
        raw.groupby(["boro", "cuisine_description", "score", "grade"], dropna=False)
        .agg(
            violation_count=("violation_code", "count"),
            critical_count=("critical_flag", lambda s: (s == "Critical").sum()),
        )
        .reset_index()
    )
    inspections_df = inspections_df.dropna(subset=["score"]).reset_index(drop=True)
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 1200
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza",
                       "Japanese", "Caribbean", "Bakery", "Coffee/Tea", "Chicken",
                       "Thai", "Vietnamese", "Ethiopian", "Peruvian"]

    violation_count = rng.poisson(lam=3, size=n)
    critical_count = rng.binomial(violation_count, 0.4)
    noise = rng.normal(0, 3, size=n)
    score = np.clip(violation_count * 7 + critical_count * 5 + noise, 0, 140).round().astype(int)

    grade = np.where(score <= 13, "A", np.where(score <= 27, "B", "C")).astype(object)
    ungraded_idx = rng.choice(n, size=int(n * 0.1), replace=False)
    grade[ungraded_idx] = None

    # Give most weight to a handful of cuisines, so the rarer ones are genuinely rare --
    # this matters later, for the high-cardinality section.
    cuisine_p = np.array([0.20, 0.14, 0.13, 0.10, 0.09, 0.08, 0.06, 0.05, 0.04, 0.03,
                          0.03, 0.02, 0.02, 0.01])
    cuisine_p = cuisine_p / cuisine_p.sum()

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_description": rng.choice(cuisines_clean, size=n, p=cuisine_p),
        "score": score,
        "grade": grade,
        "violation_count": violation_count,
        "critical_count": critical_count,
    })
    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(inspections_df):,} inspections")
inspections_df.head()

In [ ]:
inspections_df.__________

Look at those dtypes: `boro`, `cuisine_description`, and `grade` are all `object` (text). `score`,
`violation_count`, and `critical_count` are numeric, but on very different scales — `score` ranges up
into the hundreds, while `violation_count` and `critical_count` are small counts, usually under 15.

Neither of these is a problem for a human reading the table. Both are a problem for a model:

- Regression and k-NN (Weeks 7-8) do arithmetic directly on your feature values — text can't be
  subtracted or multiplied, so it has to become numbers first.
- Even once everything is numeric, a feature with a much bigger raw range can end up dominating a
  model's calculations, not because it's more important, but simply because its numbers are bigger.

### Quick recap from Lecture 1: nominal vs. ordinal

Which encoding strategy makes sense depends on the same distinction from Lecture 1's Types of Data:

- `boro` and `cuisine_description` are **nominal** — categories with no natural order. Queens isn't
  "more" or "less" than Brooklyn.
- `grade` is **ordinal** — A, B, and C really do have an order (A is better than B is better than C).

We'll treat these two cases differently below.

---

## Part 2 — Encoding Categorical Variables

### One-hot encoding, for nominal categories

**One-hot encoding** turns one categorical column into several binary (0/1) columns, one per category —
a 1 marks which category that row belongs to, everywhere else is 0.

In [ ]:
pd.__________(inspections_df[["boro"]], drop_first=False).head()

Each borough became its own column, filled with `True`/`False` (equivalent to 1/0). A row for a
Brooklyn restaurant has `boro_BROOKLYN = True` and every other borough column `False`.

`drop_first=False` keeps all five columns. In practice you'll often see `drop_first=True`, which drops
one category's column entirely (it becomes the case where every other column reads 0) — this avoids
redundant, perfectly-correlated columns for algorithms sensitive to that (linear regression among them).
Either choice is defensible; just be consistent and know which one you picked.

### The same idea, with scikit-learn's `OneHotEncoder`

`pd.get_dummies()` is convenient for a quick look, but scikit-learn's `OneHotEncoder` is what you'll
actually use in a model pipeline — it remembers the categories it was fit on, and can be told exactly
what to do if a brand new, never-before-seen category shows up in test data.

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="__________")
boro_encoded = encoder.fit_transform(inspections_df[["boro"]])

pd.DataFrame(boro_encoded, columns=encoder.get_feature_names_out(["boro"])).head()

`handle_unknown="ignore"` matters more than it looks: if your test set (or a brand new prediction
request) contains a category the encoder never saw during `fit()` -- a borough or cuisine that never
showed up in training -- this setting tells it to encode that row as all zeros instead of crashing.
Without it, a single unseen category can take down your entire prediction pipeline.

### Ordinal encoding, for categories with a real order

`grade` has a genuine order (A better than B better than C), so one-hot encoding would throw that
information away. `OrdinalEncoder` maps categories to numbers in an order **you specify explicitly** —
never let it guess, since alphabetical or first-seen order won't reliably match the order that actually
matters.

In [ ]:
grade_encoder = __________(categories=[["A", "B", "C"]])

graded_only = inspections_df.dropna(subset=["grade"]).copy()
graded_only["grade_encoded"] = grade_encoder.fit_transform(graded_only[["grade"]])

graded_only[["grade", "grade_encoded"]].drop_duplicates().sort_values("grade_encoded")

A is now 0, B is 1, C is 2 -- in that specific order, because we told the encoder that order explicitly
with `categories=[["A", "B", "C"]]`. If you'd used one-hot encoding here instead, the model would see
three unrelated columns with no sense that A and B are "closer" than A and C -- technically usable, but
throwing away real information you already have for free.

### High cardinality: when one-hot encoding gets out of hand

`cuisine_description` has a lot more distinct values than `boro` does. Let's check.

In [ ]:
cuisine_counts = inspections_df["cuisine_description"].__________()
print(f"Distinct cuisines: {len(cuisine_counts)}")
cuisine_counts

One-hot encoding this column directly would create one new column per cuisine -- and several of those
columns would represent only a handful of restaurants each. That's not just wasteful: a category that
appears three times in training gives a model almost nothing reliable to learn from, and can make some
models unstable.

A common, simple fix: group the rare categories together into a single `"Other"` bucket before encoding.

In [ ]:
MIN_COUNT = 30

common_cuisines = cuisine_counts[cuisine_counts __________ MIN_COUNT].index
inspections_df["cuisine_grouped"] = inspections_df["cuisine_description"].where(
    inspections_df["cuisine_description"].isin(common_cuisines), other="Other"
)

print(f"Categories before grouping: {inspections_df['cuisine_description'].nunique()}")
print(f"Categories after grouping:  {inspections_df['cuisine_grouped'].nunique()}")
inspections_df["cuisine_grouped"].value_counts()

---

## Part 3 — Scaling Numeric Features

In [ ]:
inspections_df[["score", "violation_count", "critical_count"]].__________()

Notice the very different ranges: `score` stretches across a much wider span than `violation_count` or
`critical_count`. That's not necessarily a problem for a decision tree or random forest (Week 10) -- they
split on one feature at a time and don't care about relative scale. But it matters a lot for:

- **k-NN (Week 8)**, which measures distance between rows -- a feature with bigger raw numbers
  automatically contributes more to that distance, whether or not it's actually more informative
- **Linear/logistic regression (Week 7)**, where features on wildly different scales can slow down or
  destabilize the fitting process

**Scaling** puts every numeric feature onto a comparable range, so no feature dominates just because of
the units it happens to be measured in.

### Standardization (z-score scaling)

This is exactly the z-score idea from Lecture 5: subtract the mean, divide by the standard deviation.
After standardizing, every feature has a mean of 0 and a standard deviation of 1.

In [ ]:
numeric_cols = ["score", "violation_count", "critical_count"]

scaler = __________()
scaled_values = scaler.fit_transform(inspections_df[numeric_cols])

scaled_df = pd.DataFrame(scaled_values, columns=[f"{c}_scaled" for c in numeric_cols])
print("Means after scaling (should be ~0):")
print(scaled_df.mean().round(4))
print("\nStandard deviations after scaling (should be ~1):")
print(scaled_df.std().round(4))
scaled_df.head()

Every column now sits on the same footing: centered at 0, spread of 1. A `violation_count` of 2 and a
`score` of 40 might now both scale to something like 0.3 -- comparable numbers, even though their original
units had nothing to do with each other.

### Min-max scaling

An alternative: squeeze every value into a fixed range, usually 0 to 1, based on the minimum and
maximum observed.

In [ ]:
minmax_scaler = __________()
minmax_values = minmax_scaler.fit_transform(inspections_df[numeric_cols])

minmax_df = pd.DataFrame(minmax_values, columns=[f"{c}_minmax" for c in numeric_cols])
print("Min after scaling (should be 0):", minmax_df.min().round(4).tolist())
print("Max after scaling (should be 1):", minmax_df.max().round(4).tolist())
minmax_df.head()

Min-max scaling is more sensitive to outliers than standardization -- a single unusually extreme value
stretches the whole 0-1 range for everyone else. Standardization is the more common default for this
reason, but min-max is useful when you need values in a strictly bounded range.

One practical note worth remembering all semester: tree-based models (decision trees, random forests,
XGBoost -- Week 10) don't need scaled features at all. Scaling only matters for distance-based and
gradient-based algorithms. It never hurts to scale, but it isn't always necessary.

---

## Part 4 — Data Leakage: The Most Important Rule in This Lecture

**Data leakage** happens when information from outside the training data -- often, information from
your test set -- accidentally influences how a model is built. It's one of the easiest ways to end up
with a model that looks great in your notebook and performs much worse in the real world, because your
evaluation was quietly cheating without you realizing it.

The rule that prevents almost all of it: **split your data first, then fit anything (scalers, encoders,
feature selection, the model itself) only on the training set.**

### The wrong way

Watch what happens if you scale *before* splitting.

In [ ]:
# WRONG -- fitting the scaler on the full dataset before splitting
leaky_scaler = StandardScaler()
leaky_scaled = leaky_scaler.__________(inspections_df[numeric_cols])

print("Mean learned by the LEAKY scaler (fit on everything):")
print(leaky_scaler.mean_.round(3))

That `mean_` was computed using **every row**, including the ones that should have been held out as a
test set. Any model trained on `leaky_scaled` afterward was implicitly given a hint about the test data's
distribution before it was ever supposed to see it. The model isn't reading test labels directly, but it's
no longer being evaluated on truly unseen data either -- the test set's influence already leaked in
through the scaler.

### The right way

In [ ]:
# RIGHT -- split first, fit the scaler on the training data only
X = inspections_df[numeric_cols]
y = inspections_df["score"]  # standing in for a real target -- more on this in Week 7

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=383)

correct_scaler = StandardScaler()
X_train_scaled = correct_scaler.__________(X_train)   # fit AND transform on train
X_test_scaled = correct_scaler.__________(X_test)         # transform ONLY, using train's fitted values

print("Mean learned by the CORRECT scaler (fit on training data only):")
print(correct_scaler.mean_.round(3))

Notice the pattern: `.fit_transform()` on the training data, `.transform()` only on the test data. The
test set is scaled *using the training set's own mean and standard deviation* -- it never contributes to
computing those numbers itself. This is the single habit that prevents most leakage: fit once, on
training data, and reuse that same fitted object everywhere else.

### Seeing the difference directly

In [ ]:
comparison = pd.DataFrame({
    "feature": numeric_cols,
    "mean (leaky, full dataset)": leaky_scaler.__________.round(3),
    "mean (correct, train only)": correct_scaler.__________.round(3),
})
comparison

The two sets of means are close but not identical here -- with a large, reasonably uniform dataset, the
gap can look small enough to shrug off. That's exactly what makes leakage dangerous: it rarely announces
itself. The same mistake gets much worse with a smaller dataset, a rare category that only appears in
your test split, or feature selection that peeks at test-set correlations before choosing which columns
to keep. The habit matters more than the size of the effect in any one example.

### A quick self-check for leakage

Ask yourself these before trusting any model's reported performance:

- Did I split my data **before** fitting any scaler, encoder, or feature selector?
- Did I call `.fit()` (or `.fit_transform()`) only on training data, everywhere in my pipeline?
- Does my test set only ever get `.transform()`, never `.fit()`?
- If I engineered a feature using some kind of aggregate (a group mean, a count, a ratio), did that
  aggregate get computed using training data only, not the full dataset?

Keep this checklist in mind for your capstone -- you're required to discuss how you prevented leakage in
your own project, and this is exactly the list to walk through when you write that section.

---

## Part 5 — Bringing It Together with Pipelines

Encoding categoricals and scaling numerics separately works, but it's easy to make a mistake -- forgetting
to refit something, or accidentally calling `.fit()` on the test set out of habit. scikit-learn's
`ColumnTransformer` and `Pipeline` bundle every preprocessing step into a single object, so `.fit()` and
`.transform()` always happen in the right place, on the right data. We'll use these properly starting
Week 11 -- this is a first, practical preview.

In [ ]:
numeric_features = ["violation_count", "critical_count"]
categorical_features = ["boro", "cuisine_grouped"]

preprocessor = ColumnTransformer(transformers=[
    ("num", __________(), numeric_features),
    ("cat", __________(handle_unknown="ignore"), categorical_features),
])

X = inspections_df[numeric_features + categorical_features]
y = inspections_df["score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=383)

# One fit call handles scaling AND encoding together, on training data only
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

print("Training features shape:", X_train_ready.shape)
print("Test features shape:    ", X_test_ready.shape)

One `preprocessor` object now handles both jobs. `fit_transform()` on `X_train` learns the scaler's
mean/std and the encoder's categories, all in one call, using training data only. `transform()` on
`X_test` reuses everything it learned -- no separate scaler and encoder to keep track of, and much harder
to accidentally leak by calling `.fit()` somewhere you shouldn't have.

---

## Part 6 — Applied on NYC 311

Same tools, back on the dataset you know from Lectures 1, 2, 3, and 5 -- and this time, prepped in a way
that sets up directly for Week 7, where `resolution_time_hours` becomes a real regression target.

### Setup

Same live-pull-with-fallback pattern as Lecture 5's Part 6.

In [ ]:
import os
SOCRATA_URL_311 = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

try:
    raw_path = os.path.expanduser("~/shared-readwrite/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_weight = np.where(days.dayofweek >= 5, 0.6, 1.0)
    day_weight = day_weight / day_weight.sum()

    day_idx = rng.choice(n_days, size=n, p=day_weight)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")

    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df[["complaint_type", "borough", "hour_filed", "resolution_time_hours"]].head()

### Split first -- before touching any preprocessing

`resolution_time_hours` is what will become the regression target in Week 7, so it isn't a feature to
scale here -- it's `y`, set aside like any other target. `hour_filed` (a genuine numeric feature) and
`complaint_type`/`borough` (nominal categoricals) are what get preprocessed.

In [ ]:
complaints_clean = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)

X_311 = complaints_clean[["complaint_type", "borough", "hour_filed"]]
y_311 = complaints_clean["resolution_time_hours"]

X_311_train, X_311_test, y_311_train, y_311_test = train_test_split(
    X_311, y_311, test_size=__________, random_state=383
)

print("Training rows:", len(X_311_train))
print("Test rows:    ", len(X_311_test))

In [ ]:
preprocessor_311 = ColumnTransformer(transformers=[
    ("num", StandardScaler(), ["hour_filed"]),
    ("cat", __________(handle_unknown="ignore"), ["complaint_type", "borough"]),
])

X_311_train_ready = preprocessor_311.fit_transform(X_311_train)
X_311_test_ready = preprocessor_311.transform(X_311_test)

print("Training features shape:", X_311_train_ready.shape)
print("Test features shape:    ", X_311_test_ready.shape)

`X_311_train_ready` and `X_311_test_ready` are now fully numeric, properly scaled and encoded, fit only
on the training split -- exactly the inputs Week 7's regression model will expect. Nothing about the model
itself has been introduced yet; this is entirely the preparation step that has to happen first.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook:
`lect06_feature_engineering_exercise.ipynb`.

---

## Part 7 — Cheat Sheet

| Task | Code |
|---|---|
| One-hot encode (quick look) | `pd.get_dummies(df[["col"]])` |
| One-hot encode (for a pipeline) | `OneHotEncoder(handle_unknown="ignore")` |
| Ordinal encode, explicit order | `OrdinalEncoder(categories=[["A","B","C"]])` |
| Standardize (z-score) | `StandardScaler().fit_transform(X_train)` |
| Min-max scale | `MinMaxScaler().fit_transform(X_train)` |
| Fit on train, apply to test | `scaler.fit_transform(X_train)` then `scaler.transform(X_test)` |
| Bundle scaling + encoding | `ColumnTransformer([...])` |
| Split before any preprocessing | `train_test_split(X, y, test_size=..., random_state=...)` |

---

## Part 8 — Key Terms

- **Feature engineering**: preparing and transforming raw data into a form a model can use.
- **One-hot encoding**: representing a categorical column as multiple binary (0/1) columns, one per
  category.
- **Ordinal encoding**: mapping ordered categories to numbers that preserve their order.
- **Cardinality**: the number of distinct values in a categorical column; "high cardinality" means many
  distinct values, often with some appearing rarely.
- **Standardization (z-score scaling)**: rescaling a numeric feature to have mean 0 and standard
  deviation 1.
- **Min-max scaling**: rescaling a numeric feature to a fixed range, usually 0 to 1.
- **Data leakage**: information from outside the training data (often the test set) improperly
  influencing model training or evaluation, making performance look better than it really is.
- **`fit()` vs. `transform()`**: `fit()` learns parameters (a mean, a set of categories) from data;
  `transform()` applies previously learned parameters to data. The leakage rule: `fit()` only on
  training data, `transform()` everywhere else.
- **`ColumnTransformer`**: a scikit-learn tool that applies different preprocessing steps to different
  columns, bundled into a single fit/transform object.